# 03 · Your First Agent — the loop, live

**Agentic AI for Health Actuaries** · IAI Seminar · 25 August 2026 · Hub: `github.com/rohanyashraj/iai-workshop`

> All data in this notebook is **hypothetical** — ABC Health is a fictional entity calibrated to plausible Indian health insurance experience, for teaching only.

**Used in:** Session 2, Part 1 (Inside the Agent Loop). 
**You will:** build a hello-agent in ~8 lines with one tool, read its ReAct trace, then break it and watch structured error handling keep it graceful.

Prereq: `GOOGLE_API_KEY` in Colab Secrets (as in notebook 01).

In [1]:
%pip install -q "agno==3.0.0" "google-genai==2.19.0"

/Users/rohanyashraj/Documents/Actuarial/2026 IFoA Workshop/25th August 2026/iai-workshop/.venv/bin/python3: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("GOOGLE_API_KEY loaded from Colab Secrets.")
except Exception:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
    print("Not running in Colab (or no secret set) - falling back to a local .env / environment variable.")

Not running in Colab (or no secret set) - falling back to a local .env / environment variable.


## §1 · The raw mechanic — what 'function calling' actually is
Before any framework: the model never runs code. It emits a **JSON request**; your runtime decides. Every guardrail you will ever build lives in that gap.

In [3]:
from google import genai
from IPython.display import Markdown, display

client = genai.Client()

def incidence_rate(condition: str) -> float:
    """Annual incidence rate (per member-year) for a critical-illness
    condition, ABC Health 2024."""
    table = {"Cardiac": 0.024, "Cancer": 0.031, "Stroke": 0.015}
    if condition not in table:
        raise ValueError(f"Unknown condition '{condition}'. Valid: {list(table)}")
    return table[condition]

# Hand the model the tool schema; it responds with a *request*, not an execution
resp = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="What is the incidence rate for Cardiac?",
    config={"tools": [incidence_rate]},   # the SDK auto-builds the schema and runs the loop
)
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(resp.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


📋 GEMINI MODEL RESPONSE


The annual incidence rate for Cardiac (ABC Health 2024) is 0.024 per member-year.


END OF MODEL RESPONSE


## §2 · Hello, agent — eight lines with Agno
`print_response(..., stream=True)` streams the answer and shows each tool call as it runs — the agent equivalent of an audit trail. Set `show_full_reasoning=True` (or `debug_mode=True` on the `Agent`) for the full trace; leave it on in development, always.

In [4]:
from agno.agent import Agent
from agno.models.google import Gemini

agent = Agent(
    model=Gemini(id="gemini-3.5-flash-lite"),
    tools=[incidence_rate],
    markdown=True,
)

agent.print_response("Compare incidence rate between Cardiac and Stroke, as a percentage difference.", stream=True, show_full_reasoning=False)


Output()

**Read your trace.** The model called the tool twice — once per condition — then computed the comparison from *tool results*, not imagination. Nobody told it to call twice: that was the ReAct loop deciding.

## §3 · Break it — structured errors the model can reason about
Ask about a condition that doesn't exist. The tool raises; Agno hands the error back **as data**; the model recovers gracefully instead of inventing a number.

In [5]:
agent.print_response("What is the incidence rate for Arthritis?", stream=True, show_full_reasoning=False)
# Expected behaviour: the agent reports that Arthritis is not a valid condition and lists the valid conditions.
# A stack trace teaches the model nothing; {"error": "Unknown condition"} is something it can reason about.


Output()

WARNING  Could not run function incidence_rate(condition=Arthritis): Unknown condition 'Arthritis'. Valid:         
         ['Cardiac', 'Cancer', 'Stroke']

ERROR    Unknown condition 'Arthritis'. Valid: ['Cardiac', 'Cancer', 'Stroke']                                     
         Traceback (most recent call last):                                                                        
           File "/Users/rohanyashraj/Documents/Actuarial/2026 IFoA Workshop/25th August                            
         2026/iai-workshop/.venv/lib/python3.12/site-packages/agno/tools/function.py", line 2219, in execute       
             result = self.function.entrypoint(**entrypoint_args, **self.arguments)                                
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                
           File "/Users/rohanyashraj/Documents/Actuarial/2026 IFoA Workshop/25th August                            
         2026/iai-workshop/.venv/lib/python3.12/site-packages/pydantic/_internal/_validate_call.py", line 40, in   
         wrapper_function                                                                                          
             return wrapper(*args, **kwargs)                                                                       
                    ^^^^^^^^^^^^^^^^^^^^^^^^                                                                       
           File "/Users/rohanyashraj/Documents/Actuarial/2026 IFoA Workshop/25th August                            
         2026/iai-workshop/.venv/lib/python3.12/site-packages/pydantic/_internal/_validate_call.py", line 137, in  
         __call__                                                                                                  
             res = self.__pydantic_validator__.validate_python(pydantic_core.ArgsKwargs(args, kwargs))             
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^             
           File "/var/folders/bx/19mx0q0j0878wgw46pf8dcl80000gn/T/ipykernel_15344/1870069718.py", line 11, in      
         incidence_rate                                                                                            
             raise ValueError(f"Unknown condition '{condition}'. Valid: {list(table)}")                            
         ValueError: Unknown condition 'Arthritis'. Valid: ['Cardiac', 'Cancer', 'Stroke']

## §4 · Exercises (5 minutes)
1. Add a second tool `average_claim_cost(condition: str) -> float` (invent plausible numbers) and ask for **expected cost** by condition — watch the agent chain both tools and do the multiplication with tool numbers.
2. Add `debug_mode=True` to the `Agent`, re-run, and see the full trace — every message, tool call and token count. That visibility is checklist question 10.
3. Ask something *outside* the tools' competence ('what will incidence be in 2027?') and observe how the agent hedges — or doesn't. What guardrail would you add?

Next: notebook 04 — the real build.
